# EDA Manual — CONJUNTO EÓLICO KAIROS

Notebook gerado automaticamente pelo **CurtailIQ Analyst Lab**.

Use este ambiente para investigar manualmente perfil horário, gaps de referência, eventos de restrição, outliers e hipóteses regulatórias.

**Arquivo de dados:** `data/conjunto_eolico_kairos_serie.csv`

**Metadados da usina:**
```json
{
  "id_estado": "CE",
  "nom_estado": "CEARA",
  "nom_subsistema": "NORDESTE",
  "potencia_mw": null,
  "pct_corte": 0.0,
  "perda_reais": 0.0,
  "score": 79.2,
  "fonte_dados": "dw.mart_eolica",
  "nom_usina": "CONJUNTO EÓLICO KAIROS",
  "gerado_em": "2026-06-22T01:12:46Z",
  "n_linhas_exportadas": 576,
  "csv": "data/conjunto_eolico_kairos_serie.csv"
}
```

In [ ]:
# Setup Pyodide/JupyterLite: carrega libs analíticas se necessário
try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
except Exception:
    import micropip
    await micropip.install(['pandas', 'matplotlib'])
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
pd.set_option('display.max_columns', 80)


In [ ]:
DATA_PATH = 'data/conjunto_eolico_kairos_serie.csv'
df = pd.read_csv(DATA_PATH, parse_dates=['din_instante'])
df = df.sort_values('din_instante').reset_index(drop=True)
df['gap_mw'] = df['val_geracaoreferencia'].fillna(0) - df['val_geracao'].fillna(0)
df['corte_mw'] = np.where(df['cod_razaorestricao'].notna(), df['gap_mw'].clip(lower=0), 0)
df['hora'] = df['din_instante'].dt.hour + df['din_instante'].dt.minute / 60
df['dia'] = df['din_instante'].dt.date
df['mes'] = df['din_instante'].dt.to_period('M').astype(str)
df.head()


In [ ]:
# Resumo executivo da série
resumo = pd.Series({
    'inicio': df['din_instante'].min(),
    'fim': df['din_instante'].max(),
    'pontos_30min': len(df),
    'geracao_mwh': df['val_geracao'].sum()/2,
    'referencia_mwh': df['val_geracaoreferencia'].sum()/2,
    'corte_mwh': df['corte_mw'].sum()/2,
    'pct_corte': (df['corte_mw'].sum() / df['val_geracaoreferencia'].sum()) if df['val_geracaoreferencia'].sum() else np.nan,
    'intervalos_restritos': int(df['cod_razaorestricao'].notna().sum()),
})
resumo


In [ ]:
# Série temporal: geração, referência e corte
ax = df.set_index('din_instante')[['val_geracao','val_geracaoreferencia']].tail(800).plot(figsize=(14,5), lw=1)
df.set_index('din_instante')['corte_mw'].tail(800).plot(ax=ax, lw=1, color='crimson', alpha=.7)
ax.set_title('Geração x Referência x Corte (últimos pontos)')
ax.set_ylabel('MW')
ax.grid(True, alpha=.25)


In [ ]:
# Perfil horário médio: quando a usina mais sofre corte?
perfil = df.groupby('hora', as_index=False).agg(
    ger_mw=('val_geracao','mean'), ref_mw=('val_geracaoreferencia','mean'), corte_mw=('corte_mw','mean')
)
ax = perfil.plot(x='hora', y=['ger_mw','ref_mw','corte_mw'], figsize=(12,4), marker='o')
ax.set_title('Perfil horário médio')
ax.set_ylabel('MW médio')
ax.grid(True, alpha=.25)


In [ ]:
# Quebra por razão de restrição
razao = (df[df['cod_razaorestricao'].notna()]
    .groupby(['cod_razaorestricao','cod_origemrestricao'], dropna=False)
    .agg(intervalos=('din_instante','count'), corte_mwh=('corte_mw', lambda s: s.sum()/2))
    .reset_index()
    .sort_values('corte_mwh', ascending=False))
razao


In [ ]:
# Outliers simples por z-score do gap
gap = df['gap_mw'].fillna(0)
df['gap_z'] = (gap - gap.mean()) / (gap.std() or 1)
outliers = df.loc[df['gap_z'].abs() >= 3, ['din_instante','val_geracao','val_geracaoreferencia','gap_mw','corte_mw','cod_razaorestricao','gap_z']]
outliers.sort_values('gap_z', ascending=False).head(30)


## Ideias de investigação manual
- Comparar eventos `ENE` vs `CNF/REL` e revisar elegibilidade regulatória.
- Verificar horários recorrentes de sobreoferta e restrições locais.
- Cruzar picos de `gap_mw` com notícias/regulação/indisponibilidades.
- Exportar recortes de eventos para dossiê de pleito.
